# 从价格函数出发求解 Recursive Competitive Equilibrium

## TL;DR

这份 notebook 重新 frame 一个 **price-operator / dual formulation of RCE**：不把一般均衡首先看成高维 policy-function approximation，而是看成定义在分布空间上的均衡定价算子固定点。

要把 household/firm problems 作为独立的动态规划直接求解，最小的 recursive information 通常是

$$
\mathcal P:S_t\mapsto p_t(\cdot),\qquad
\mathcal T:(S_t,z_{t+1})\mapsto S_{t+1}.
$$

其中 $S_t$ 在异质性模型里包含完整分布。$\mathcal P$ 决定当期预算约束，但 Bellman expectation 还需要 $\mathcal T$ 来确定下一期 continuation state。也可以不用显式学习 $\mathcal T$，而是在每次 Bellman/policy update 中由 population best responses 重新 push forward；但这样 household problem 与 aggregate transition 成为一个 coupled inner fixed point，不能说是“给定 $\mathcal P$ 后直接解一个独立 household problem”。

三个结论：

1. 普通 RCE 包含 pricing rule 与 aggregate-state transition；算法可以联合学习二者，也可以只学习 price rule、在内层重新求 transition。
2. 在 Krusell--Smith 中，当期 Cobb--Douglas factor prices 可能只依赖总资本，但未来价格通常仍依赖完整分布，因为非线性储蓄政策会使同均值分布产生不同的 $K'$。
3. 在 household 与 firm 双边异质性的模型中，共同 pricing operator 仍是自然经济接口，但 expectation formation 还需要一个 self-consistent transition 或等价的 recursive continuation-price law。


## 0. 目标、范围与关键假设

### 目标

我们要回答四个问题：

1. 没有异质性时，RCE 通常怎样迭代 aggregate law 和 price function？
2. 引入 Krusell--Smith 异质性以后，price function 的定义域怎样从低维 aggregate state 变成分布空间？
3. 得到什么样的价格对象，才足以直接求解 KS household problem？
4. household 和 firm 都异质时，怎样定义并计算 recursive competitive equilibrium？

### 关键假设

- Agents 是 price takers，单个 agent 不影响 aggregate distribution。
- “价格”采用广义定义：除了工资、利率和商品价格，还包括 transfers、tax schedules、dividends、markups、asset prices，以及所有直接进入个体预算约束或收益函数的共同均衡对象。
- 给定完整价格体系及其条件未来规律时，household 和 firm problems 有可选择的最优解。若存在多重均衡，还需要额外的 equilibrium-selection rule，pricing operator 才是单值的。
- 计算中的 neural operator 总是有限离散化；所谓 infinite-dimensional 指 continuum limit 下定义域是 measures/functions，并要求算法随 agent number 或 distribution grid refinement 收敛。


## 1. 一般 RCE：正常的计算方法是什么？

先考虑一个 aggregate state $A_t$ 和 aggregate shock $z_t$。代表性 household 的个体状态记作 $x_t$，firm 的生产技术给出 factor demand。一个 stationary RCE 通常包含：

$$
\{V(x,A,z),\;g(x,A,z),\;\mathcal P(A,z),\;\mathcal T(A,z,z')\}.
$$

这里

- $\mathcal P(A,z)$ 是当期 equilibrium price rule；
- $\mathcal T(A,z,z')=A'$ 是 aggregate state law of motion；
- household 给定 $\mathcal P$ 和 $\mathcal T$ 解动态规划；
- 个体政策加总后必须重新产生 $\mathcal T$；
- 由 firms' FOCs 或 market clearing 产生的价格必须重新等于 $\mathcal P$。

$\mathcal T$ 是 RCE 定义中需要满足的一致性对象，但不一定需要独立猜测或学习。若给定 policies 后能够精确 push forward aggregate state，则对一个候选 $\mathcal P^n$ 可以在内层联合求解

$$
g^n=BR(\mathcal P^n,\mathcal T^n),\qquad
\mathcal T^n=\operatorname{PushForward}(g^n).
$$

然后计算 market residual 并只更新 pricing operator：

$$
\mathcal P^n\longrightarrow(g^n,\mathcal T^n)\longrightarrow
\mathcal M^n\longrightarrow\mathcal P^{n+1}.
$$


### 1.1 两种常见的 price iteration

**方法 A：先猜 aggregate law，再由结构公式得到价格。**

在 Cobb--Douglas RBC 中，若猜测 $K'=G(K,z)$ 和劳动 $N=N(K,z)$，则 firms' FOCs 给出

$$
r(K,z)+\delta=\alpha z\left(\frac{K}{N(K,z)}\right)^{\alpha-1},\qquad
w(K,z)=(1-\alpha)z\left(\frac{K}{N(K,z)}\right)^\alpha.
$$

这时程序表面上迭代的是 $G,N$，但经济上等价于迭代由它们诱导的 price rule。项目中的 `RBC.ipynb` 正属于这种做法。

**方法 B：直接猜 price rule，并用 excess demand 更新。**

对于每个 aggregate state $S$，定义 market residual

$$
\mathcal M(S,p)=D(S,p)-S^s(S,p).
$$

价格可以通过 tâtonnement 或 Newton 更新：

$$
p^{n+1}(S)=p^n(S)+\eta\,\mathcal M(S,p^n(S)),
$$

或者

$$
p^{n+1}(S)=p^n(S)-[\partial_p\mathcal M(S,p^n(S))]^{-1}\mathcal M(S,p^n(S)).
$$

关键是：每一次价格猜测都可能要求重新计算 agents' responses；在双边异质性模型中，这个嵌套会很昂贵。


In [1]:
using Statistics, Printf

const ALPHA = 0.36
const DELTA = 0.025
const PRODUCTIVITY = 1.0
const LABOR = 1.0

capital_demand(r) = LABOR * (ALPHA * PRODUCTIVITY / (r + DELTA))^(1 / (1 - ALPHA))
capital_supply(r) = 12.0 + 50.0 * (r - 0.02)

function solve_toy_market(; initial_rate=0.04, step_size=0.0004, max_iterations=300)
    rate = initial_rate
    history = NamedTuple[]
    for iteration in 1:max_iterations
        demand = capital_demand(rate)
        supply = capital_supply(rate)
        excess_demand = demand - supply
        push!(history, (; iteration, r=rate, demand, supply, excess_demand))
        abs(excess_demand) < 1e-10 && break
        rate += step_size * excess_demand
    end
    return history
end

price_history = solve_toy_market()
solution = last(price_history)
@printf("iterations = %d\n", solution.iteration)
@printf("equilibrium r = %.8f\n", solution.r)
@printf("capital demand = %.8f, capital supply = %.8f\n", solution.demand, solution.supply)
@printf("market residual = %.3e\n", solution.excess_demand)


iterations = 157
equilibrium r = 0.04403895
capital demand = 13.20194752, capital supply = 13.20194752
market residual = 8.633e-11


上面的 toy calculation 只想说明函数迭代的结构。真正的动态模型中，`capital_supply(r)` 不是一个已知公式，而是 household 在当前价格和未来 price law 下的最优储蓄加总。因此，每个 $r$ guess 背后可能嵌套一次完整的动态规划。


## 2. 引入 HA：Krusell--Smith 中 price function 怎样变化？

令 household state 为 $(a,e)$，其中 $a$ 是资产，$e$ 是 idiosyncratic employment/productivity state。横截面分布为

$$
\Gamma_t\in\mathcal M(\mathcal A\times\mathcal E).
$$

完整 aggregate state 变成

$$S_t=(\Gamma_t,z_t).$$

因此精确 RCE 的 price rule 不再只是 $r(K,z),w(K,z)$ 的普通低维函数，而是定义在 measure space 上的 functional：

$$
\mathcal P:(\Gamma,z)\mapsto\{r[\Gamma,z],w[\Gamma,z],\tau[\Gamma,z],\ldots\}.
$$

对于标准 Cobb--Douglas KS，当期 factor prices 具有结构分解

$$
K[\Gamma]=\int a\,d\Gamma(a,e),
$$

$$
r[\Gamma,z]+\delta=\alpha z\left(\frac{K[\Gamma]}{L[\Gamma,z]}\right)^{\alpha-1},\qquad
w[\Gamma,z]=(1-\alpha)z\left(\frac{K[\Gamma]}{L[\Gamma,z]}\right)^\alpha.
$$

所以当期 $r,w$ 可能只通过少数 moments 依赖 $\Gamma$。但这不意味着这些 moments 足以预测未来价格：$K'$ 是非线性 individual policy 对完整分布的积分。


In [2]:
function factor_prices(assets; z=PRODUCTIVITY, labor=LABOR)
    capital = mean(assets)
    rental_rate = ALPHA * z * (capital / labor)^(ALPHA - 1) - DELTA
    wage = (1 - ALPHA) * z * (capital / labor)^ALPHA
    return (; capital, rental_rate, wage)
end

# 两个 empirical distributions 有完全相同的均值 K=10，但离散程度不同。
tight_distribution = vcat(fill(8.0, 500), fill(12.0, 500))
dispersed_distribution = vcat(fill(1.0, 500), fill(19.0, 500))

# 一个刻意简化的非线性储蓄规则，用来展示分布形状对 K' 的影响。
saving_rule(a) = 0.88 * a + 0.80 * sqrt(a)

current_tight = factor_prices(tight_distribution)
current_dispersed = factor_prices(dispersed_distribution)
next_tight = factor_prices(saving_rule.(tight_distribution))
next_dispersed = factor_prices(saving_rule.(dispersed_distribution))

@assert isapprox(current_tight.capital, current_dispersed.capital; atol=1e-12)
@assert isapprox(current_tight.rental_rate, current_dispersed.rental_rate; atol=1e-12)

@printf("current: K = %.4f, r = %.6f, w = %.6f (both distributions)\n",
        current_tight.capital, current_tight.rental_rate, current_tight.wage)
@printf("next from tight distribution:     K' = %.4f, r' = %.6f, w' = %.6f\n",
        next_tight.capital, next_tight.rental_rate, next_tight.wage)
@printf("next from dispersed distribution: K' = %.4f, r' = %.6f, w' = %.6f\n",
        next_dispersed.capital, next_dispersed.rental_rate, next_dispersed.wage)


current: K = 10.0000, r = 0.057471, w = 1.466155 (both distributions)
next from tight distribution:     K' = 11.3170, r' = 0.051193, w' = 1.532934
next from dispersed distribution: K' = 10.9436, r' = 0.052847, w' = 1.514527


这个例子区分了两个对象：

- **当期 structural pricing functional** 可能很简单，甚至只依赖 $K[\Gamma]$；
- **equilibrium price law** 必须知道最优 policy 如何把整个 $\Gamma$ 推到 $\Gamma'$，因此通常不能只由当期 $K$ 精确决定。

标准 Krusell--Smith 用 perceived law

$$\log K'=a_z+b_z\log K$$

近似这个高维映射，并检验 approximate aggregation。我们的 idea 是不预先假定只有 $K$ 重要，而是直接学习定义在完整分布上的 equilibrium pricing operator / price-law operator。


### 2.1 得到什么 function 后，可以直接求解 KS household problem？

KS household 的 Bellman equation 是

$$
V(a,e;\Gamma,z)=
\max_{a'\geq\underline a}
\left\{u(c)+\beta\mathbb E\left[V(a',e';\Gamma',z')\mid e,z\right]\right\},
$$

subject to

$$
c+a'=(1+r[\Gamma,z])a+w[\Gamma,z]e+T[\Gamma,z,e].
$$

这里的 expectation 需要知道

$$
\Gamma'=\mathcal T[\Gamma,z,z'].
$$

即使 $\mathcal P$ 已知于所有可能分布，它只告诉我们“到达某个 $\Gamma'$ 后价格是多少”，却没有告诉 household “哪一个 $\Gamma'$ 会发生”。因此有三种计算表示：

**A. Distribution-state representation**

$$
\boxed{\mathcal P[\Gamma,z]\quad\text{and}\quad
\mathcal T[\Gamma,z,z']=\Gamma'}.
$$


**B. Recursive price-state representation**

构造一个包含完整 continuation information 的 price state $\chi$：

$$
p=\mathcal D(\chi),\qquad
\chi'=\mathcal L(\chi,z').
$$

这里 $\mathcal L$ 必须把 household 带到一个能够继续生成所有未来条件价格的 next price state；只预测一个 $p_{t+1}$ 或 $\mathbb E_t p_{t+1}$ 通常不够。

**C. Price-only outer representation**

只参数化 $\mathcal P$，但每次评价它时在内层联合求解

$$
g^{\mathcal P}=BR_H(\mathcal P,\mathcal T_{\mathcal P}),\qquad
\mathcal T_{\mathcal P}=\operatorname{PushForward}(g^{\mathcal P}).
$$

这种表示不需要单独学习 $\mathcal T$，但 household Bellman equation 不能脱离 aggregate transition 单独求解。


### 2.2 只学习 $\mathcal P$ 时，expectation formation 变成内层固定点

对候选 operator $\mathcal P_\theta$，household Euler/Bellman problem 中的下一期价格直接写成

$$
p_{t+1}(z')
=\mathcal P_\theta\!\left[
\operatorname{PushForward}\!\left(\Gamma_t,g^{\mathcal P_\theta},z'\right),z'
\right].
$$

因此可以定义记号

$$
\mathcal T_{\mathcal P_\theta}
:=\operatorname{PushForward}(g^{\mathcal P_\theta}),\qquad
\mathcal Q_{\mathcal P_\theta}
:=\mathcal P_\theta\circ\mathcal T_{\mathcal P_\theta},
$$

$\mathcal T_{\mathcal P_\theta}$ 和 $\mathcal Q_{\mathcal P_\theta}$ 可以不被独立参数化，但必须在形成 Bellman expectation 之前由 inner policy/distribution fixed point 算出来。换言之：外层可以只学习 $\mathcal P_\theta$，但 household problem 的有效输入绝不只是当期 $p_t$。


## 3. KS 的 price-operator 求解算法

令 $BR_H$ 表示 household solver，例如 EGM、policy iteration 或 Euler-equation network。有两个不同的 price-operator 算法。

**方案 A：联合学习 pricing operator 与 aggregate transition**

1. 学习 $p=\mathcal P_\theta[\Gamma,z]$ 和 $\widehat\Gamma'=\mathcal T_\psi[\Gamma,z,z']$。
2. household 用 $\mathcal P_\theta[\widehat\Gamma',z']$ 形成 expectation，并解 $g_H=BR_H(\mathcal P_\theta,\mathcal T_\psi)$。
3. 用 $g_H$ 得到真实 push-forward $\Gamma'_{BR}$。
4. 同时最小化 market clearing 与 transition consistency：

$$
\mathcal L_T=\mathbb E_{S,z'}\|\mathcal T_\psi[S,z']-\operatorname{PushForward}(\Gamma,g_H,z')\|^2.
$$

这最接近标准 KS：传统 KS 学习低维 $K'=G(K,z,z')$，这里把它推广为完整分布上的 operator。

**方案 B：只学习 pricing operator，内层重算 transition**

1. 学习当期 $p=\mathcal P_\theta[\Gamma,z]$。
2. 在每个训练 state 内联合迭代 $g^{\mathcal P_\theta}=BR_H(\mathcal P_\theta,\mathcal T_{\mathcal P_\theta})$ 与 $\mathcal T_{\mathcal P_\theta}=\operatorname{PushForward}(g^{\mathcal P_\theta})$。
3. 内层收敛后，只用 market-clearing residual 更新 $\theta$：

$$
\mathcal L_{MC}=\mathbb E_S\|\mathcal M(S,\mathcal P_\theta[S],g_H)\|^2,
$$


方案 B 的 equilibrium unknown 更少，但每次评价 $\mathcal P_\theta$ 都嵌套一个 forward--backward equilibrium solve。方案 A 多学习一个 operator，却把 expectation formation amortize 掉，更可能具有计算优势。也可以不学习 distribution transition，而直接学习 Markov-sufficient recursive price state $(\mathcal D_\omega,\mathcal L_\psi)$。


### 3.1 不应学习已经知道的 Cobb--Douglas 公式

推荐把 operator 写成

$$
(\Gamma,z)
\xrightarrow{\text{measure neural operator}}
A_\theta[\Gamma,z]
\xrightarrow{\text{structural decoder}}
p.
$$

$A_\theta$ 只输出结构中未知的 equilibrium aggregates 或 multipliers；资本积分、Cobb--Douglas FOCs、zero profit、no-arbitrage 等已知关系直接 hard-code。比如

$$
\mathcal P_\theta[\Gamma,z]
=D_{CD}\!\left(K[\Gamma],L_\theta[\Gamma,z],z\right).
$$

标准 KS 中当前 factor-price operator 很低秩，真正值得学习的通常是 distribution-dependent continuation price law。异质企业、融资约束、endogenous markups、资产价格或连续商品价格 schedule 会产生更丰富的 pricing operator。


## 4. 双边异质性模型

现在令 household individual state 为 $x\in X_H$，firm individual state 为 $y\in X_F$，两个横截面分布分别为

$$\mu_t^H\in\mathcal M(X_H),\qquad \mu_t^F\in\mathcal M(X_F).$$

aggregate state 是

$$S_t=(\mu_t^H,\mu_t^F,z_t).$$

广义价格体系可以是有限向量，也可以是一条由市场坐标 $j\in\mathcal J$ 索引的 schedule：

$$
\mathcal P:S_t\mapsto p_t(\cdot)\in C(\mathcal J).
$$

例如 $p(\cdot)$ 可以同时包含 wage schedule、capital rental rate、bond/equity prices、goods prices、demand schedules、transfers 和 taxes。只要某个共同对象直接影响 agents' payoffs 或 constraints，就应包含在 generalized price system 中。


### 4.1 双边异质性 Recursive Competitive Equilibrium 的定义

给定 aggregate shock transition，一个 RCE 是

$$
\mathcal E=
\{V_H,g_H,V_F,g_F,\mathcal P,\mathcal T_H,\mathcal T_F\},
$$

满足以下条件。对一个候选 $\mathcal P$，先把由两边 best responses 推出的下一期状态记作

$$
S'_{\mathcal P}(z')=
\left(\mathcal T_H^{\mathcal P}(S,z'),\mathcal T_F^{\mathcal P}(S,z'),z'\right).
$$

**1. Household optimality**

$$
g_H^{\mathcal P}(x,S)\in BR_H\!\left(x;\mathcal P[S],\{\mathcal P[S'_{\mathcal P}(z')]\}_{z'}\right).
$$

**2. Firm optimality**

$$
g_F^{\mathcal P}(y,S)\in BR_F\!\left(y;\mathcal P[S],\{\mathcal P[S'_{\mathcal P}(z')]\}_{z'}\right).
$$

**3. Distribution consistency**

$$
\mathcal T_H^{\mathcal P}=\operatorname{PushForward}_H(\mu^H,g_H^{\mathcal P},z'),\qquad
\mathcal T_F^{\mathcal P}=\operatorname{PushForward}_F(\mu^F,g_F^{\mathcal P},z').
$$

这些是 policies 与 idiosyncratic transition kernels 诱导的 push-forward / Kolmogorov forward operators。

**4. Market clearing for every market $j$**

$$
\mathcal M_j(S,p)
=\int d_j(x,g_H,p)\,d\mu^H(x)
 -\int s_j(y,g_F,p)\,d\mu^F(y)=0,\qquad\forall j.
$$

**5. Future prices are derived, not learned**

$$
p'(z')=\mathcal P[S'_{\mathcal P}(z')].
$$

这是一条定义恒等式，不是额外的 equilibrium condition，也不需要单独的 law-of-motion network。因此双边模型可以被压缩成 pricing-operator root problem：

$$
\boxed{
\mathcal P
\rightarrow (g_H^{\mathcal P},g_F^{\mathcal P},\mathcal T_H^{\mathcal P},\mathcal T_F^{\mathcal P})
\rightarrow\mathcal M_{\mathcal P},\qquad
\mathcal M_{\mathcal P}=0
}.
$$


### 4.2 为什么 pricing operator 是双边模型的自然接口？

直接的 policy-first representation 是

$$
g_H:X_H\times\mathcal M_H\times\mathcal M_F\times Z\to A_H,
$$

$$
g_F:X_F\times\mathcal M_H\times\mathcal M_F\times Z\to A_F.
$$

household network 必须直接理解 firm distribution，firm network 也必须直接理解 household distribution。price-first representation 则是

$$
(\mu^H,\mu^F,z)\xrightarrow{\mathcal P}
\text{common current prices},
$$

$$
g_H^{\mathcal P}=BR_H(x;\mathcal P,\mathcal T_{\mathcal P}),\qquad
g_F^{\mathcal P}=BR_F(y;\mathcal P,\mathcal T_{\mathcal P}).
$$

未来价格通过两边 policies 推出的 $S'_{\mathcal P}$ 再送回同一个 $\mathcal P$ 得到。两个 population problems 在每次给定共同价格路径时可以并行求解，但 policies 与两个 distribution transitions 仍需在内层达到一致。两种分布的相互影响只通过共同的 equilibrium price system 和由其诱导的 aggregate transition 传递。


## 5. 双边模型的建议求解器：operator predictor + equilibrium corrector

如果目标是让 household 和 firm problems 可以独立、并行地形成 expectations，推荐同时使用 price predictor 与 transition predictor：

1. **Price and transition prediction**

   $$p^{(0)}=\mathcal P_\theta[S],\qquad
   \widehat S'=\mathcal T_\psi[S,z'].$$

2. **Parallel best responses using forecast continuation prices**

   $$g_H=BR_H(\mathcal P_\theta,\mathcal T_\psi),\qquad
   g_F=BR_F(\mathcal P_\theta,\mathcal T_\psi),$$

   其中 continuation prices 是 $\mathcal P_\theta[\widehat S']$。

3. **Push-forward correction**

   用 $g_H,g_F$ 推动两个分布得到 $S'_{BR}$，并令 $\mathcal T_\psi[S,z']\approx S'_{BR}$。

4. **Market-clearing correction**

   $$p^{(1)}=p^{(0)}-[\partial_p\mathcal M(S,p^{(0)})]^{-1}\mathcal M(S,p^{(0)}),$$

   再把 $p^{(1)}$ distill 回 $\mathcal P_\theta$。

这里 $\mathcal T_\psi$ 不是任意 reduced-form forecast：它必须等于两边最优 policies 诱导的 push-forward。如果直接学习 recursive continuation-price state，可以用相同思路把 transition consistency 写在 price-state space 中。


In [3]:
# 一个与具体模型无关的软件接口草图。各 callback 可以由 EGM、firm solver、
# neural best-response operator 或 exact market equations 实现。

struct BilateralState{H,F,Z}
    household_distribution::H
    firm_distribution::F
    aggregate_shock::Z
end

function equilibrium_operator_residuals(
    state, next_shock;
    price_operator, transition_operator,
    household_best_response, firm_best_response,
    push_distributions_forward, market_residual, transition_residual
)
    prices = price_operator(state)
    forecast_next_state = transition_operator(state, next_shock)
    continuation_prices = price_operator(forecast_next_state)
    household_policy = household_best_response(state, prices, continuation_prices)
    firm_policy = firm_best_response(state, prices, continuation_prices)

    clearing = market_residual(state, prices, household_policy, firm_policy)
    implied_next_state = push_distributions_forward(
        state, household_policy, firm_policy, next_shock
    )
    transition_consistency = transition_residual(forecast_next_state, implied_next_state)

    return (; clearing, transition_consistency, implied_next_state)
end

println("Defined a model-agnostic bilateral price-and-transition operator residual interface.")


Defined a model-agnostic bilateral price-and-transition operator residual interface.


## 6. 如何表示真正的 infinite-dimensional operator？

固定长度的 moments 向量并不等于学习完整分布上的 functional。可考虑：

- 一维或二维分布：density、CDF 或 quantile function 网格上的 DeepONet/FNO；
- empirical measures：共享 point encoder 加 permutation-invariant aggregation；
- 双边分布：household 与 firm 分别使用 measure encoder，再通过 cross-attention 或 market-specific decoder 生成价格；
- continuum price schedule：输入 market coordinate $j$，查询 $p_\theta[\mu^H,\mu^F,z](j)$，而不是固定输出一个价格网格。

必须做 discretization convergence：增加 household agents、firm agents 或 distribution grid points 后，价格、policies、Euler residual 和 market residual 应稳定。否则只是对某个固定有限样本向量进行拟合。


## 7. 计算比较与可证伪的研究假设

应至少比较三种求解器：

1. **Traditional root solver**：每个 aggregate state 上从头迭代 market-clearing prices。
2. **Policy-first neural solver**：直接学习 household/firm policies，把 Euler/KKT 和 market clearing 作为 residuals。
3. **Price/transition-operator predictor--corrector**：学习 $\mathcal P,\mathcal T$，用 best-response push-forward 修正 transition，并做少量 market-clearing corrections。

统一报告：

- wall-clock time、GPU/CPU memory、market-root iterations；
- household Euler/KKT residual、firm optimality residual；
- 每个市场的 clearing residual；
- transition prediction 与 best-response push-forward 的一致性 residual；
- distribution-grid refinement 和 population-size refinement；
- ergodic-set 内和 distribution-shift/OOD 测试；
- 参数变化、政策优化和 estimation 中的 amortized runtime。

最核心的可证伪假设是：

> 在双边异质性模型中，equilibrium pricing operator 比两个直接依赖双方完整分布的 policy operators 更平滑、更容易泛化；用它初始化市场出清可以显著减少每个 aggregate state 上重复求两边 best responses 的次数。


## 8. Takeaways 与下一步

- RCE 的经济学定义天然支持 price-operator formulation：价格体系是 decentralize allocations 的共同 dual object。
- KS 中当期 Cobb--Douglas price functional 往往已知且低秩；困难的是完整分布通过 nonlinear policies 影响未来 distribution 与 continuation prices。
- 当前 pricing functional $\mathcal P(S)$ 本身不足以形成 Bellman expectation；还需要 $S'=\mathcal T(S,z')$，或等价的 Markov-sufficient recursive continuation-price law。
- 双边异质性下，$\mathcal P[\mu^H,\mu^F,z]$ 把 household 和 firm problems 模块化，是这个方法最有吸引力的应用。
- 最稳健的计算实现可能是 structural pricing layer + distribution/price-law transition operator + push-forward consistency + 少量 market-clearing correction。

建议下一步先选择一个当前价格不能完全由单一 aggregate moment 决定、但仍有可靠 benchmark solver 的双边模型。先实现 exact-root baseline，再逐步加入 price operator；这样可以清楚识别速度提升到底来自 price prediction、population decomposition，还是 best-response amortization。
